# 00b. Selección de segmentos nuevos para EP3 (validación externa sellada)

**Para correr en Google Colab, no en este repositorio.** Toma segmentos que EP2 nunca evaluó para probar la solución final una sola vez (plan v3, P9). Los pasos exactos están en `docs/ep3/GUIA_VALIDACION_EXTERNA.md`.

**Reglas de esta selección**
- Usa **solo metadatos** (hora del día, ubicación, número de frames, número de detecciones) y nunca `sin_camara` ni resultados de modelos. El orden de candidatos se calculó antes, sin descargar nada, y está en `docs/ep3/plan_seleccion_ep3.json` (sha256 en `plan_seleccion_ep3.sha256`): **el notebook exige subir el plan y su hash y usa su orden**; el recalculado en Colab solo se compara.
- **Mismo pool que EP2** (`00b_seleccion_segmentos_ep2.ipynb`): sin lluvia (793) → al menos 190 frames (786) → `n_detecciones` entre los percentiles 2.5 y 97.5 calculados sobre esos 786 (2.027 a 33.688; pool de 746) → **solo después** se excluyen los 91 segmentos que EP2 ya evaluó (655 candidatos). El notebook se detiene si el embudo no da 793, 786 y 746.
- **Cuotas fijas de segmentos ÚTILES por estrato** (`time_of_day x location`), declaradas en la celda de parámetros:

| Estrato | Cuota de útiles | Candidatos disponibles |
|---|---|---|
| Day / location_sf | 8 | 313 |
| Day / location_phx | 4 | 191 |
| Day / location_other | 2 | 81 |
| Night / location_sf | 3 | 23 |
| Night / location_phx | 2 | 9 |
| Dawn/Dusk / location_sf | 3 | 15 |
| Dawn/Dusk / location_phx | 3 | 22 |
| Dawn/Dusk / location_other | 1 | 1 |

  Total de la cuota: 26. `Night / location_other` no tiene candidatos.
- **Evaluación secuencial por estrato** en el orden aleatorio fijado por la semilla 42 (`numpy.random.default_rng`), hasta completar la cuota o agotar el estrato (queda registrado cuántos «faltan»). Los estratos se recorren por rondas para que el tope global de **55 evaluaciones** no deje sin evaluar a los últimos. **Meta mínima: 15 útiles.**
- Un candidato **se descarta** si: la asociación 2D está vacía; tiene menos de 50 asociaciones; las asociaciones no cruzan con `lidar_box` (0 positivos); tiene menos de **30 positivos** (cajas LiDAR con par de cámara); o el etiquetado es ralo (menos de 5.5 % de las cajas LiDAR de los frames etiquetados tiene par). Son los criterios de EP2.
- Descarga solo `lidar_box` y `camera_to_lidar_box_association` (no imágenes).

**Por qué cuotas fijas y no proporcionales.** El desarrollo (EP2) tuvo cuotas por estrato, no proporcionales al tamaño; una asignación proporcional daría ~90 % de segmentos de día. **Hay sobremuestreo de San Francisco de día** (8 útiles) porque ahí están las 2 celdas con hueco de EP2 (2 de los 7 segmentos seleccionados de ese estrato): más segmentos de ese estrato permiten ver si el detector vuelve a marcar celdas. La elección usa solo metadatos y se declara antes de abrir datos.

**Limitación declarada: la noche queda mal representada.** De noche quedan 32 candidatos (23 en San Francisco, 9 en Phoenix, 0 en otras ubicaciones) y en EP2 solo 12 de 42 candidatos nocturnos resultaron útiles (29 %): lo probable es que las cuotas de noche no se completen. El resultado no representa bien la noche.

**Nota de honestidad sobre el criterio de etiquetado.** Mide qué tan completo está el etiquetado 2D del segmento (pares sobre cajas LiDAR de todos los tipos, en frames etiquetados), igual que en EP2. No usa la tasa de `sin_camara` de peatones y ciclistas, pero sí deja fuera segmentos con etiquetado 2D muy escaso: el detector y la regla solo se prueban en segmentos con etiquetado denso (sesgo de selección documentado en el informe, §3).

**Licencia.** Los datos del Waymo Open Dataset son de uso no comercial y no se redistribuyen. El `.tar.gz` que produce este notebook es solo para su uso local en el curso.

## 1. Parámetros y funciones de selección (fijados antes de mirar nada)
La celda siguiente es una **copia exacta** de `src/waymo_evaluacion/ep3/seleccion_nuevos.py` (una prueba del repositorio comprueba que coinciden). Aquí están las cuotas, los umbrales y la lógica del orden.

In [ ]:
"""Selección de segmentos nuevos para la validación externa (F18-B). Funciones PURAS (sin red, sin bucket, sin escribir archivos).

El notebook `notebooks/00b_seleccion_segmentos_ep3.ipynb` (Colab) incluye el código fuente de estas mismas funciones (se copia con `inspect.getsource`;
una prueba comprueba que coinciden) y `docs/ep3/plan_seleccion_ep3.json` guarda el orden de candidatos que producen, calculado SIN descargar nada
con `docs/seleccion_segmentos/inventario_segmentos.csv` y `registro_verificacion_ep2.csv`. La elección usa solo metadatos y se declara antes de abrir datos.
"""
from __future__ import annotations

from typing import Any, Callable, Iterable

import numpy as np
import pandas as pd

SEMILLA = 42
MIN_FRAMES = 190                        # igual que el 00b de EP2
PERCENTILES_DETECCIONES = (2.5, 97.5)   # se calculan sobre los segmentos que pasan los filtros anteriores (igual que EP2)
MIN_ASOCIACIONES = 50
MIN_POSITIVOS = 30                      # igual que EP2 (motivo «pocos positivos»)
MIN_TASA_ETIQUETADA = 5.5               # % de cajas LiDAR de frames etiquetados con par de cámara
META_UTILES = 15
TOPE_EVALUACIONES = 55
# cuotas de segmentos ÚTILES por estrato (time_of_day, location); Night / location_other no tiene candidatos
CUOTAS: dict[tuple[str, str], int] = {
    ("Day", "location_sf"): 8, ("Day", "location_phx"): 4, ("Day", "location_other"): 2,
    ("Night", "location_sf"): 3, ("Night", "location_phx"): 2,
    ("Dawn/Dusk", "location_sf"): 3, ("Dawn/Dusk", "location_phx"): 3, ("Dawn/Dusk", "location_other"): 1,
}


def nombre_estrato(tod: str, loc: str) -> str:
    return f"{tod} / {loc}"


def embudo(inventario: pd.DataFrame, evaluados_ep2: Iterable[str]) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Pool elegible como en el 00b de EP2 y SOLO DESPUÉS se excluyen los segmentos que EP2 ya evaluó.

    Pasos: sin lluvia -> al menos MIN_FRAMES frames -> n_detecciones entre los percentiles (calculados sobre el resultado del paso anterior)
    -> se excluyen los evaluados por EP2. (El paso «existe lidar_box y asociación» de EP2 depende del bucket: el notebook lo cuenta en Colab.)
    """
    pasos: dict[str, Any] = {"inventario": int(len(inventario))}
    c = inventario[inventario["weather"] != "rain"]
    pasos["sin_lluvia"] = int(len(c))
    c = c[c["n_frames"] >= MIN_FRAMES]
    pasos[f"n_frames_>={MIN_FRAMES}"] = int(len(c))
    bajo, alto = c["n_detecciones"].quantile([PERCENTILES_DETECCIONES[0] / 100, PERCENTILES_DETECCIONES[1] / 100])
    pool = c[(c["n_detecciones"] >= bajo) & (c["n_detecciones"] <= alto)]
    pasos["n_detecciones_entre_percentiles"] = int(len(pool))
    pasos["limite_inferior_detecciones"] = float(bajo)
    pasos["limite_superior_detecciones"] = float(alto)
    ev = set(evaluados_ep2)
    pasos["evaluados_ep2_dentro_del_pool"] = int(pool["segment_id"].isin(ev).sum())
    candidatos = pool[~pool["segment_id"].isin(ev)].copy()
    pasos["candidatos_sin_los_evaluados_por_ep2"] = int(len(candidatos))
    return candidatos, pasos


def orden_por_estrato(candidatos: pd.DataFrame, cuotas: dict[tuple[str, str], int] = CUOTAS, semilla: int = SEMILLA) -> dict[str, list[str]]:
    """Orden aleatorio fijo de los candidatos de cada estrato: ids ordenados y permutados con ``default_rng(semilla)`` (un generador nuevo por estrato,
    así cambiar la cuota de un estrato no altera el orden de los demás)."""
    out: dict[str, list[str]] = {}
    for (tod, loc) in cuotas:
        ids = sorted(candidatos.loc[(candidatos["time_of_day"] == tod) & (candidatos["location"] == loc), "segment_id"])
        out[nombre_estrato(tod, loc)] = [str(s) for s in np.random.default_rng(semilla).permutation(np.array(ids, dtype=object))] if ids else []
    return out


def decidir(m: dict[str, Any]) -> tuple[str, str]:
    """Aplica los criterios de «segmento útil» (mismos que EP2) a las métricas medidas de un candidato.

    ``m``: ``asoc_filas``, ``lidar_disponible`` (bool), ``positivos`` (cajas LiDAR con par de cámara) y ``tasa_etiquetado_pct``.
    Devuelve ``(resultado, motivo)``; resultado es «util» o «descartado».
    """
    if m.get("asoc_filas", 0) == 0:
        return "descartado", "asociacion vacia"
    if m["asoc_filas"] < MIN_ASOCIACIONES:
        return "descartado", f"pocas asociaciones ({m['asoc_filas']} < {MIN_ASOCIACIONES})"
    if not m.get("lidar_disponible", True):
        return "descartado", "lidar_box no disponible"
    if m.get("positivos", 0) == 0:
        return "descartado", "las asociaciones no cruzan con lidar_box"
    if m["positivos"] < MIN_POSITIVOS:
        return "descartado", f"pocos positivos ({m['positivos']} < {MIN_POSITIVOS})"
    if m["tasa_etiquetado_pct"] < MIN_TASA_ETIQUETADA:
        return "descartado", f"etiquetado ralo ({m['tasa_etiquetado_pct']:.1f}% < {MIN_TASA_ETIQUETADA}% en frames etiquetados)"
    return "util", ""


def seleccionar(orden: dict[str, list[str]], cuotas: dict[tuple[str, str], int], evaluar: Callable[[str], dict[str, Any]],
                tope: int = TOPE_EVALUACIONES, previos: dict[str, dict[str, Any]] | None = None) -> list[dict[str, Any]]:
    """Evaluación secuencial por estrato, en el orden aleatorio fijo, hasta completar la cuota o agotar el estrato, con un tope global de evaluaciones.

    Los estratos se recorren por rondas (un candidato de cada estrato no completo por ronda, en el orden de ``cuotas``) para que el tope global no
    deje sin evaluar a los últimos estratos; dentro de cada estrato el orden es secuencial. ``previos`` permite reanudar (segmento -> registro ya medido).
    Devuelve el registro de evaluaciones (cada fila con ``segment_id``, ``estrato``, ``resultado`` y ``motivo``).
    """
    previos = previos or {}
    claves = [nombre_estrato(*h) for h in cuotas]
    cuota = {nombre_estrato(*h): q for h, q in cuotas.items()}
    util = {k: 0 for k in claves}
    pos = {k: 0 for k in claves}
    registro: list[dict[str, Any]] = []
    while len(registro) < tope:
        avanzo = False
        for k in claves:
            if len(registro) >= tope:
                break
            if util[k] >= cuota[k] or pos[k] >= len(orden.get(k, [])):
                continue
            seg = orden[k][pos[k]]
            pos[k] += 1
            r = dict(previos[seg]) if seg in previos else dict(evaluar(seg))
            r.update(segment_id=seg, estrato=k)
            registro.append(r)
            avanzo = True
            if r["resultado"] == "util":
                util[k] += 1
        if not avanzo:
            break
    return registro


def resumen_cuotas(registro: list[dict[str, Any]], cuotas: dict[tuple[str, str], int], orden: dict[str, list[str]]) -> pd.DataFrame:
    """Por estrato: cuota, evaluados, útiles, faltan, candidatos que quedaron sin evaluar y la razón (cuota completa / estrato agotado / tope global)."""
    filas = []
    for h, q in cuotas.items():
        k = nombre_estrato(*h)
        ev = [r for r in registro if r["estrato"] == k]
        u = sum(r["resultado"] == "util" for r in ev)
        quedan = len(orden.get(k, [])) - len(ev)
        razon = "cuota completa" if u >= q else ("estrato agotado" if quedan <= 0 else "tope global de evaluaciones")
        filas.append({"estrato": k, "cuota": q, "candidatos_en_el_estrato": len(orden.get(k, [])), "evaluados": len(ev), "utiles": u, "faltan": max(q - u, 0), "razon": razon})
    return pd.DataFrame(filas)


def construir_plan(inventario: pd.DataFrame, evaluados_ep2: Iterable[str], hashes_fuentes: dict[str, str] | None = None) -> dict[str, Any]:
    """Plan de selección pre-declarado: parámetros, cuotas, embudo y el orden de candidatos por estrato (sin descargar nada)."""
    cand, pasos = embudo(inventario, evaluados_ep2)
    orden = orden_por_estrato(cand)
    return {
        "version": "plan_seleccion_ep3_v1",
        "nota": "Elección con solo metadatos (inventario y registro de EP2); declarada antes de abrir datos. No usa sin_camara.",
        "parametros": {"semilla": SEMILLA, "min_frames": MIN_FRAMES, "percentiles_detecciones": list(PERCENTILES_DETECCIONES), "min_asociaciones": MIN_ASOCIACIONES, "min_positivos": MIN_POSITIVOS,
                       "min_tasa_etiquetada_pct": MIN_TASA_ETIQUETADA, "meta_utiles": META_UTILES, "tope_evaluaciones": TOPE_EVALUACIONES,
                       "generador": "numpy.random.default_rng(42).permutation(ids ordenados), un generador nuevo por estrato",
                       "recorrido": "por rondas entre estratos no completos; secuencial dentro de cada estrato"},
        "cuotas_utiles": {nombre_estrato(*h): q for h, q in CUOTAS.items()},
        "total_cuota": int(sum(CUOTAS.values())),
        "embudo": pasos,
        "candidatos_por_estrato": {k: len(v) for k, v in orden.items()},
        "orden_por_estrato": orden,
        "fuentes_sha256": hashes_fuentes or {},
    }

In [ ]:
import hashlib, json, os, subprocess, tarfile, time
from pathlib import Path
import numpy as np
import pandas as pd

BUCKET = "gs://waymo_open_dataset_v_2_0_1/training"
TRABAJO = Path("/content/ep3_nuevos")          # carpeta de trabajo en Colab
SALIDA = TRABAJO / "salida"
(SALIDA / "lidar_box").mkdir(parents=True, exist_ok=True)
(SALIDA / "camera_to_lidar_box_association").mkdir(parents=True, exist_ok=True)
(TRABAJO / "tmp").mkdir(parents=True, exist_ok=True)
print("cuota total de utiles:", sum(CUOTAS.values()), "| tope de evaluaciones:", TOPE_EVALUACIONES, "| meta:", META_UTILES)

## 2. Entorno y acceso al bucket (Colab)
Hay que haber aceptado la licencia del Waymo Open Dataset con su cuenta de Google. `gsutil` ya viene en Colab.

In [ ]:
try:
    from google.colab import auth, files
    auth.authenticate_user()
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
    print("No es Colab: asegure gsutil autenticado y los archivos de metadatos en RUTA_META.")
print(subprocess.run(["gsutil", "version"], capture_output=True, text=True).stdout.splitlines()[:1])

## 3. Archivos que hay que subir (los 5 son OBLIGATORIOS)
Los tres CSV de `docs/seleccion_segmentos/` (`inventario_segmentos.csv` con los 798 segmentos, `registro_verificacion_ep2.csv` con los 91 evaluados y `diagnostico_lluvia_ep2.csv`) **y el plan sellado** `docs/ep3/plan_seleccion_ep3.json` con su `plan_seleccion_ep3.sha256`. Sin los cinco, o si el hash del plan no coincide, el notebook se detiene antes de tocar el bucket.

In [ ]:
RUTA_META = TRABAJO / "meta"
RUTA_META.mkdir(exist_ok=True)
REQUERIDOS = ["inventario_segmentos.csv", "registro_verificacion_ep2.csv", "diagnostico_lluvia_ep2.csv", "plan_seleccion_ep3.json", "plan_seleccion_ep3.sha256"]
if EN_COLAB and not all((RUTA_META / n).exists() for n in REQUERIDOS):
    print("Suba los 5 archivos:", ", ".join(REQUERIDOS))
    for nombre, contenido in files.upload().items():
        (RUTA_META / nombre).write_bytes(contenido)
faltan = [n for n in REQUERIDOS if not (RUTA_META / n).exists()]
assert not faltan, f"FALTAN ARCHIVOS OBLIGATORIOS en {RUTA_META}: {faltan}. Vuelva a ejecutar esta celda y suba los 5."
inv = pd.read_csv(RUTA_META / "inventario_segmentos.csv", dtype={"segment_id": str})
reg = pd.read_csv(RUTA_META / "registro_verificacion_ep2.csv", dtype={"segment_id": str})
assert len(inv) == 798 and len(reg) == 91, (len(inv), len(reg))
print(f"inventario {len(inv)} | ya evaluados en EP2 {len(reg)}")

# el plan sellado: existe, su sha256 coincide y trae la version esperada
ruta_plan, ruta_hash = RUTA_META / "plan_seleccion_ep3.json", RUTA_META / "plan_seleccion_ep3.sha256"
HASH_PLAN = hashlib.sha256(ruta_plan.read_bytes()).hexdigest()
assert HASH_PLAN == ruta_hash.read_text().split()[0], "EL PLAN SUBIDO NO COINCIDE CON SU sha256: no siga (el sellado no vale)"
plan = json.loads(ruta_plan.read_text(encoding="utf-8"))
assert plan.get("version") == "plan_seleccion_ep3_v1", plan.get("version")
print("OK: plan sellado verificado, sha256", HASH_PLAN[:16], "... version", plan["version"])

## 4. Embudo, candidatos y orden por estrato (solo metadatos)
El orden que se USA es el del plan sellado (verificado por hash). El notebook también lo recalcula con las versiones de numpy y pandas de Colab y lo compara: si difiere, se usa igualmente el del plan y se imprime una advertencia visible. Si el embudo difiere en conteos, se detiene.

In [ ]:
# El paso «existe lidar_box y asociacion» de EP2 depende del bucket: se cuenta aqui y debe dar 793 (todos los sin lluvia).
def listar(componente):
    r = subprocess.run(["gsutil", "ls", f"{BUCKET}/{componente}/*.parquet"], capture_output=True, text=True)
    assert r.returncode == 0, f"gsutil ls {componente} fallo: {r.stderr[:300]}"
    return {Path(l).stem for l in r.stdout.splitlines() if l.endswith(".parquet")}
con_ambos = listar("lidar_box") & listar("camera_to_lidar_box_association")
sin_lluvia = set(inv.loc[inv["weather"] != "rain", "segment_id"])
n_con_ambos = len(sin_lluvia & con_ambos)
print("sin lluvia:", len(sin_lluvia), "| con lidar_box y asociacion:", n_con_ambos)
assert n_con_ambos == 793, f"el paso con lidar_box y asociacion deberia dar 793 como en EP2 y da {n_con_ambos}: NO seguir y reportarlo"

candidatos, pasos = embudo(inv, reg["segment_id"])
print(json.dumps(pasos, indent=1))
assert (pasos["sin_lluvia"], pasos[f"n_frames_>={MIN_FRAMES}"], pasos["n_detecciones_entre_percentiles"]) == (793, 786, 746), "el embudo no coincide con EP2: NO seguir y reportarlo"
assert pasos["candidatos_sin_los_evaluados_por_ep2"] == 655, pasos
# el embudo calculado aqui debe tener los mismos conteos que el del plan
for k, v in plan["embudo"].items():
    if isinstance(v, int):
        assert pasos[k] == v, f"EL EMBUDO DIFIERE DEL PLAN en {k}: Colab {pasos[k]} contra plan {v}. NO seguir y reportarlo."

orden_calculado = orden_por_estrato(candidatos)
orden = plan["orden_por_estrato"]                      # el que se USA: el del plan sellado
ORDEN_COINCIDE = (orden_calculado == orden)
if ORDEN_COINCIDE:
    print("OK: el orden recalculado con numpy", np.__version__, "y pandas", pd.__version__, "coincide con el del plan sellado.")
else:
    print("!" * 78)
    print("ADVERTENCIA: el orden recalculado en este Colab (numpy", np.__version__, ") NO coincide con el del plan sellado.")
    print("Se usa el orden DEL PLAN (verificado por hash). Reporte esta diferencia: queda registrada en verificacion_colab.json.")
    print("!" * 78)
    from IPython.display import HTML, display as _display
    _display(HTML("<div style='background:#FDECEA;border:2px solid #B42318;padding:10px;font-size:16px'><b>ADVERTENCIA:</b> el orden recalculado difiere del plan sellado; se usa el del plan.</div>"))
ids_plan = {s for v in orden.values() for s in v}
assert ids_plan <= set(candidatos["segment_id"]), "el plan trae segmentos que no estan entre los candidatos: NO seguir"
print({k: len(v) for k, v in orden.items()})

## 5. Evaluación de cada candidato
Se descargan solo las dos tablas necesarias. El criterio de «etiquetado ralo» se calcula como en EP2: cajas LiDAR con par de cámara (clave `frame + laser_object_id`) divididas por las cajas LiDAR de los frames que tienen al menos un par; los **positivos** son las cajas LiDAR con par. **No se calcula ni se mira `sin_camara` de peatones y ciclistas.** Es reanudable si Colab se reinicia: `seleccionar` repite la misma secuencia y salta lo ya medido.

In [ ]:
def gcopiar(ruta, destino):
    r = subprocess.run(["gsutil", "-q", "cp", ruta, str(destino)], capture_output=True, text=True)
    return r.returncode == 0

def evaluar(seg):
    t = TRABAJO / "tmp"
    for viejo in ("asoc.parquet", "lidar.parquet"):           # evita arrastrar archivos del candidato anterior
        (t / viejo).unlink(missing_ok=True)
    m = dict(segment_id=seg, asoc_filas=0, lidar_disponible=True, lidar_filas=np.nan, frames_totales=np.nan, frames_etiquetados=np.nan, positivos=0, tasa_etiquetado_pct=np.nan, archivo_asoc_ausente=False)
    if not gcopiar(f"{BUCKET}/camera_to_lidar_box_association/{seg}.parquet", t / "asoc.parquet"):
        m["archivo_asoc_ausente"] = True
    else:
        a = pd.read_parquet(t / "asoc.parquet", columns=["key.frame_timestamp_micros", "key.laser_object_id"])
        m["asoc_filas"] = len(a)
        if len(a) >= MIN_ASOCIACIONES:                   # como en EP2: solo se baja lidar_box si la asociacion tiene filas suficientes
            if not gcopiar(f"{BUCKET}/lidar_box/{seg}.parquet", t / "lidar.parquet"):
                m["lidar_disponible"] = False
            else:
                l = pd.read_parquet(t / "lidar.parquet", columns=["key.frame_timestamp_micros", "key.laser_object_id"])
                par = l.merge(a.drop_duplicates().assign(_p=1), on=["key.frame_timestamp_micros", "key.laser_object_id"], how="left")
                par["_p"] = par["_p"].fillna(0)
                etiquetado = par.groupby("key.frame_timestamp_micros")["_p"].transform("max") > 0
                en_frames = par[etiquetado]
                m.update(lidar_filas=len(l), frames_totales=l["key.frame_timestamp_micros"].nunique(), frames_etiquetados=int(par.loc[etiquetado, "key.frame_timestamp_micros"].nunique()),
                         positivos=int(par["_p"].sum()), tasa_etiquetado_pct=round(100 * float(en_frames["_p"].sum()) / max(len(en_frames), 1), 3))
    m["resultado"], m["motivo"] = decidir(m)
    return m

In [ ]:
REGISTRO = TRABAJO / "registro_verificacion_ep3.csv"
hechos = pd.read_csv(REGISTRO, dtype={"segment_id": str}) if REGISTRO.exists() else pd.DataFrame()
previos = {r["segment_id"]: r for r in hechos.to_dict("records")}
if previos:
    print(f"Reanudando: {len(previos)} candidatos ya medidos en {REGISTRO.name}")
meta = inv.set_index("segment_id")
t0 = time.time()

def evaluar_y_guardar(seg):
    r = evaluar(seg)
    r.update(time_of_day=meta.loc[seg, "time_of_day"], location=meta.loc[seg, "location"], n_frames_inventario=int(meta.loc[seg, "n_frames"]))
    if r["resultado"] == "util":                      # se conserva solo lo util
        for comp, arch in (("lidar_box", "lidar.parquet"), ("camera_to_lidar_box_association", "asoc.parquet")):
            (TRABAJO / "tmp" / arch).replace(SALIDA / comp / f"{seg}.parquet")
    pd.DataFrame([r]).to_csv(REGISTRO, mode="a", header=not REGISTRO.exists(), index=False)
    print(f"{seg[:12]} {r['resultado']:9s} {r['motivo']}  ({time.time()-t0:.0f} s)")
    return r

registro = seleccionar(orden, CUOTAS, evaluar_y_guardar, TOPE_EVALUACIONES, previos)
reg3 = pd.DataFrame(registro)

## 6. Resultado de la selección

In [ ]:
n_util = int((reg3["resultado"] == "util").sum())
resumen = resumen_cuotas(registro, CUOTAS, orden)
print(f"evaluados {len(reg3)} (tope {TOPE_EVALUACIONES}) | utiles {n_util} (meta {META_UTILES}, cuota {sum(CUOTAS.values())}) | descartados {len(reg3) - n_util}")
display(resumen)
print(reg3["motivo"].str.replace(r"\(.*\)", "", regex=True).value_counts().to_string())
if n_util < META_UTILES:
    print(f"ATENCION: solo {n_util} utiles (< {META_UTILES}). No se reajusta nada con los datos: vea docs/ep3/GUIA_VALIDACION_EXTERNA.md (que hacer si salen menos de 15).")

## 7. Exportar el paquete
Se escribe la lista de segmentos útiles, el registro completo (incluye descartados), el resumen por estrato («faltan»), los parámetros y `verificacion_colab.json` (hash del plan, versiones de numpy y pandas de Colab, embudo, cuotas y si el orden recalculado coincidió), y se empaqueta todo en un `.tar.gz` con su sha256. Descárguelo y llévelo al repositorio (por ejemplo a `data/01_raw_nuevos/`).

In [ ]:
utiles = reg3[reg3["resultado"] == "util"][["segment_id", "estrato", "time_of_day", "location", "n_frames_inventario", "asoc_filas", "lidar_filas", "positivos", "tasa_etiquetado_pct"]]
utiles.to_csv(SALIDA / "segmentos_nuevos_ep3.csv", index=False)
reg3.to_csv(SALIDA / "registro_verificacion_ep3.csv", index=False)
resumen.to_csv(SALIDA / "resumen_cuotas_ep3.csv", index=False)
params = dict(plan_sha256=HASH_PLAN, semilla=SEMILLA, meta_utiles=META_UTILES, tope_evaluaciones=TOPE_EVALUACIONES, min_frames=MIN_FRAMES, percentiles=list(PERCENTILES_DETECCIONES), min_asociaciones=MIN_ASOCIACIONES,
              min_positivos=MIN_POSITIVOS, min_tasa_etiquetada_pct=MIN_TASA_ETIQUETADA, cuotas={f"{a} / {b}": q for (a, b), q in CUOTAS.items()}, embudo=pasos, bucket=BUCKET,
              usa_sin_camara=False, evaluados=int(len(reg3)), utiles=int(len(utiles)), segmentos_utiles=sorted(utiles["segment_id"]), orden_por_estrato=orden)
(SALIDA / "parametros_seleccion_ep3.json").write_text(json.dumps(params, indent=1), encoding="utf-8")
import platform, sys as _sys
verificacion = dict(plan_sha256=HASH_PLAN, plan_version=plan["version"], numpy=np.__version__, pandas=pd.__version__, python=platform.python_version(), plataforma=platform.platform(),
                    embudo=pasos, embudo_coincide_con_el_plan=True, con_lidar_box_y_asociacion=n_con_ambos, cuotas={f"{a} / {b}": q for (a, b), q in CUOTAS.items()},
                    orden_recalculado_coincide_con_el_plan=bool(ORDEN_COINCIDE), orden_usado="el del plan sellado", evaluados=int(len(reg3)), utiles=int(len(utiles)),
                    sha256_csv={n: hashlib.sha256((RUTA_META / n).read_bytes()).hexdigest() for n in REQUERIDOS[:3]})
(SALIDA / "verificacion_colab.json").write_text(json.dumps(verificacion, indent=1), encoding="utf-8")
print(json.dumps(verificacion, indent=1))
paquete = TRABAJO / "segmentos_nuevos_ep3.tar.gz"
with tarfile.open(paquete, "w:gz") as tar:
    tar.add(SALIDA, arcname="segmentos_nuevos_ep3")
h = hashlib.sha256(paquete.read_bytes()).hexdigest()
print("paquete:", paquete, f"({paquete.stat().st_size/1e6:.1f} MB)\nsha256:", h)
if EN_COLAB:
    files.download(str(paquete))

## 8. Qué sigue (en el repositorio, en este orden; detalle en `docs/ep3/GUIA_VALIDACION_EXTERNA.md`)
1. Descomprima el paquete en `data/01_raw_nuevos/` (queda `segmentos_nuevos_ep3/`).
2. `.venv\Scripts\python scripts/ep3/evaluar_sellado.py congelar --lista data/01_raw_nuevos/segmentos_nuevos_ep3/segmentos_nuevos_ep3.csv` (congela lista y modelos con sha256, **antes** de abrir datos).
3. `.venv\Scripts\python scripts/ep3/ingesta_nuevos.py --entrada data/01_raw_nuevos/segmentos_nuevos_ep3 --lista <la misma lista>`.
4. `.venv\Scripts\python scripts/ep3/evaluar_sellado.py evaluar --lista <la misma lista>` (una sola vez; crea el candado).
Nada de lo que salga se usa para elegir ni reajustar.